# ReFuelEU optimisation — runs

Minimise the discounted total surplus loss over 10 design variables (5 reference years ×
2 pathways) subject to G1–G6, for one case at a time. Set `CASE` below and run the
notebook; run it again with a different `CASE` for the next one. Nothing is shared
between cases, so several can run at once in separate kernels.

| `CASE` | biomass to aviation | efficiency gain | published as |
|---|---|---|---|
| `main` | 9.9 % | 1.35 %/yr | reference case |
| `B5` | 5 % | 1.35 %/yr | biomass sensitivity |
| `B75` | 7.5 % | 1.35 %/yr | biomass sensitivity |
| `B15` | 15 % | 1.35 %/yr | biomass sensitivity |
| `pess` | 9.9 % | 0.91 %/yr | pessimistic technology roadmap |

Everything else is identical across the five, which is why they live in one `CASES`
table in `optimisation_runs.py` rather than in five near-duplicate notebooks.

**The constraints** (G2–G6 in `constraints_rte.py`, G1 from `models_optim_complex`):

* **G1** carbon budget
* **G2** blend completeness, χ_B + χ_E ≤ 100
* **G3/G4** biomass and electricity availability
* **G5/G6** ramp-up, the least constraining of a rate and a volume limit (Eq. 12)

## 0. Setup

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import gemseo as gm
from aeromaps.utils.functions import custom_logger_config

import optimisation_runs as R

warnings.filterwarnings("ignore")
custom_logger_config(gm.configure_logger())

CASE = "main"  # main | B5 | B75 | B15 | pess

print(f"{CASE}: {R.CASES[CASE]}")

## 1. The two fixed-mandate references

Neither is an optimisation — a single MDA each, seconds rather than minutes. `fossil` is
the BAU the paper measures everything against; `refueleu` is the regulation as written.
Both are needed by `02_results.ipynb`.

In [ ]:
R.run_reference(CASE, kind="fossil")
R.run_reference(CASE, kind="refueleu")

## 2. The carbon-budget sweep

The published notebooks ran each budget independently from a hand-pasted start point,
and several of the tighter budgets simply failed — they are commented out as "not
possible" in `run_opt_B05.ipynb` and friends.

`run_sweep` walks the budgets **from loosest to tightest and starts each optimisation
from the previous one's optimum**. Tightening the budget moves the optimum smoothly, so
its predecessor is a far better guess than any fixed vector: SLSQP starts just outside
the new feasible set instead of somewhere unrelated to it. An infeasible run is not
propagated — the chain carries the last feasible design forward instead.

Two consequences worth knowing:

* Budgets that were unreachable before may now converge, so the full ladder 3.8 → 2.0 is
  attempted for every case rather than pre-pruned. Read the `feasible` column; a budget
  that is genuinely below the system's floor will still fail, and that is a result.
* The sequence is a chain. Re-running one budget in the middle with a different start
  point gives a legitimately different local optimum.

Results are written to `results/` **as each run finishes**, and a budget already on disk
is skipped with its optimum read back from the HDF — so an interrupted sweep resumes
cleanly. Delete a file to force that one to re-run.

**Cost.** One MDA takes a few seconds and each SLSQP iteration costs eleven of them
(finite differences over 10 variables), so a case is tens of minutes to a few hours.
Start with `max_iter=3` to confirm the wiring before committing to a full sweep.

**When a run fails.** Three different things are meant by that, and the sweep treats
them differently:

* **Ends infeasible** (SLSQP finds no feasible point, or `max_iter` cuts it off before
  it does). The run is saved anyway and the sweep goes on, but the design is *not*
  passed to the next budget — the chain carries the last feasible one instead. The
  `status` column says `ran, INFEASIBLE`.
* **Raises** — an `MDAConvergenceError`, say. Nothing is saved for that budget, the
  exception text lands in `status`, and the ladder continues from the last feasible
  design. One blow-up no longer costs a whole overnight sweep. Pass
  `stop_on_error=True` if you would rather have the traceback.
* **Already on disk.** Skipped, *including* when it was infeasible — on the same start
  point it would only reproduce itself. The summary marks those
  `on disk, INFEASIBLE - delete to retry`; retrying is worth it only with a larger
  `max_iter` or a different `x0`, both of which mean deleting the two files first.

The returned frame always covers the whole ladder, so it reads the same whether the
sweep ran from scratch or resumed.

In [ ]:
summary = R.run_sweep(CASE, max_iter=50)
summary

## 3. What came out

`min CO₂` is the last row, and it is the paper's second problem rather than a budget:
the carbon budget constraint *becomes* the objective and is dropped from the constraint
set, leaving G2–G6. It is the left-hand end of the trade-off curve — the least CO₂ the
system can reach at any cost — so it is run once per case, as `opt_<case>_mincarb`.

Being the tight end, it inherits the last feasible budget's optimum like every other
rung: squeezing the budget pushes the mandate towards the same resource and ramp-up
limits that bind the min-CO₂ solution, so that is the closest start available.
`MIN_CARBON_START` is only the fallback for running it standalone.

One difference from the published notebooks: they solved this one with NLOPT's MMA.
nlopt is not installed in this environment, so both objectives use SLSQP.

In [ ]:
frames = []
for budget in ["mincarb"] + [
    R.budget_tag(b) for b in [2.0, 2.2, 2.4, 2.6, 2.8, 3.0, 3.2, 3.4, 3.6, 3.8]
]:
    path = R.RESULTS_DIR / f"opt_{CASE}_{budget}.json"
    if not path.exists():
        continue
    vector, floats = R.load(path)
    frames.append(
        {
            "budget": budget,
            "CO2 share of world budget (%)": floats["carbon_budget_consumed_share"]
            / R.EU_ASK_SHARE,
            "surplus loss (Bn EUR)": vector["cumulative_total_surplus_loss_discounted"].loc[2050]
            / 1e9,
            "cumulative CO2 (Gt)": vector["cumulative_co2_emissions"].loc[2050],
            "biofuel 2050 (%)": vector["generic_biofuel_share_dropin_fuel"].loc[2050],
            "electrofuel 2050 (%)": vector["generic_electrofuel_share_dropin_fuel"].loc[2050],
            "RPK 2050 (Bn)": vector["rpk"].loc[2050] / 1e9,
        }
    )

pd.DataFrame(frames).round(2)

In [ ]:
# Optimised mandates across the sweep, against the regulation. Compare with Figure 7.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
budgets = [
    b
    for b in [2.0, 2.2, 2.4, 2.6, 2.8, 3.0, 3.2, 3.4, 3.6, 3.8]
    if (R.RESULTS_DIR / f"opt_{CASE}_{R.budget_tag(b)}.json").exists()
]
colours = plt.cm.viridis(np.linspace(0.15, 0.9, len(budgets)))

for budget, colour in zip(budgets, colours):
    vector, _ = R.load(R.RESULTS_DIR / f"opt_{CASE}_{R.budget_tag(budget)}.json")
    for ax, pathway in zip(axes, ["generic_biofuel", "generic_electrofuel"]):
        ax.plot(
            R.OPTIM_YEARS,
            vector[f"{pathway}_share_dropin_fuel"].loc[R.OPTIM_YEARS],
            "-o",
            ms=3,
            color=colour,
            label=f"{budget}",
        )

for ax, (name, refueleu) in zip(
    axes,
    [
        ("Biofuel", R.REFUELEU_MANDATE["biofuel"]),
        ("Electrofuel", R.REFUELEU_MANDATE["electrofuel"]),
    ],
):
    ax.plot(R.OPTIM_YEARS, refueleu, "--", color="black", lw=1.5, label="ReFuelEU")
    ax.set_title(name)
    ax.set_xlabel("Year")
    ax.grid(alpha=0.3)

axes[0].set_ylabel("Drop-in fuel share (%)")
axes[1].legend(title="World budget (%)", fontsize=8, ncol=2, loc="upper left")
fig.suptitle(f"Optimised blending mandate — {R.CASES[CASE]['label']}")
plt.tight_layout()

## 4. The other cases, and running them in parallel

Set `CASE` at the top and re-run, or drive the whole thing from a shell — the module is
executable, and the five cases share nothing:

```sh
for case in main B5 B75 B15 pess; do
    poetry run python optimisation_runs.py $case > log_$case.txt 2>&1 &
done
```

Five processes, one core each, five sweeps in the wall time of the slowest. **This is
the level at which the problem parallelises**, and nothing below it is worth the
trouble:

* The budgets within a case are a continuation chain, so they are sequential by
  construction. Running them independently means giving that up and cold-starting each.
* Parallel finite differences inside a single run do not pay. GEMSEO refuses threads
  outright — *"all workers shall be different objects"*, since the eleven perturbed
  points share one MDA object. Processes do run, but measured **436 s against 112 s**
  serial over three SLSQP iterations: each point ships the whole 106-discipline chain to
  a worker, and the workers cannot see the memo that `share_mda_across_functions`
  installs, whose ~7× saving is worth more than the parallelism it would buy.

Once all five are on disk, `02_results.ipynb` draws the paper's figures. It needs, per
case: the ten budgets, `mincarb`, and both references.

Individual runs can be redone one at a time, outside the continuation — this is how you
retry a budget that came out infeasible:

```python
R.run_optimisation(CASE, budget=2.6, max_iter=120,
                   x0={"biofuel": [...], "electrofuel": [...]})
R.run_optimisation(CASE, budget=None)  # min CO2
```

`00_migration_validation.ipynb` section 3 dissects any run from its HDF.